# 🎲 Advanced AI Gamemaster - OpenEnv RL Training with Unsloth

This notebook implements **Multi-Dimensional Reward GRPO** to train a self-improving Gamemaster.

### Bleeding Edge Mechanics:
1. **System 2 Chain-of-Thought:** The model is forced to explain its logic before acting.
2. **Spatial World Modeling:** The model tracks Player vs Monster (x,y) locations.
3. **Durable Recall:** The model must remember a 'Rusty Key' acquired 50 turns ago.
4. **Tension Management:** The model is rewarded for keeping player HP between 10-30%.
5. **Self-Generated Challenges:** The model designs the next monster's stats.

In [ ]:
# 1. SETUP: Clone your project from GitHub to Colab
!git clone https://github.com/http-pruthvi/gamemaster.git
import os
os.chdir('gamemaster/gamemaster_env')

# 2. INSTALL: Dependencies
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install openenv-core pydantic

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import re
import json
from client import GamemasterEnv
from models import GamemasterAction

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    use_gradient_checkpointing="unsloth",
)

In [ ]:
ENV_URL = "https://Pruthvi1762-gamemaster-env.hf.space/api"

def get_env_result(generated_text):
    try:
        json_match = re.search(r'\{.*\}', generated_text, re.DOTALL)
        if not json_match: return None
        action_data = json.loads(json_match.group(0))
        action = GamemasterAction(**action_data)
        
        with GamemasterEnv(base_url=ENV_URL).sync() as client:
            client.reset()
            return client.step(action)
    except: return None

def rule_reward(prompts, completions, **kwargs):
    return [get_env_result(c[0]["content"]).observation.metadata.get("rule_accuracy", -1.0) 
            if get_env_result(c[0]["content"]) else -2.0 for c in completions]

def spatial_reward(prompts, completions, **kwargs):
    return [get_env_result(c[0]["content"]).observation.metadata.get("spatial", 0.0) 
            if get_env_result(c[0]["content"]) else 0.0 for c in completions]

def pacing_reward(prompts, completions, **kwargs):
    return [get_env_result(c[0]["content"]).observation.metadata.get("pacing", 0.0) 
            if get_env_result(c[0]["content"]) else 0.0 for c in completions]

In [ ]:
SYSTEM_PROMPT = """
You are an AI Gamemaster. Enforce rules strictly.
OBSERVATION:
Contains player action, dice roll, HP, locations.

RULES:
1. Spatial: Player must be at the SAME location as the monster to attack.
2. Combat: If attacking and roll >= 10, hit. Apply damage. If roll < 10, miss. 0 damage.
3. Recall: If player tries to unlock an iron door, check if 'Rusty Key' is in Inventory.
4. Next Level: If monster dies, generate next_monster_name, hp, and dmg.

You MUST respond with a JSON object:
{
  "gm_logic": "Step-by-step reasoning...",
  "narrative_response": "Story text...",
  "target_to_damage": "goblin" or null,
  "damage_amount": integer,
  "item_to_give": null,
  "next_monster_name": null,
  "next_monster_hp": null,
  "next_monster_dmg": null
}
"""

scenarios = [
    {"observation": "Player: 'I attack!' | Dice Roll: 15 | Player Loc: [1,2] | Monster Loc: [1,2]"},
    {"observation": "Player: 'I attack!' | Dice Roll: 15 | Player Loc: [0,0] | Monster Loc: [1,2]"},
    {"observation": "Player: 'I use the Rusty Key!' | Inv: ['Rusty Key']"},
]

dataset_dict = {"prompt": []}
for s in scenarios * 30: 
    dataset_dict["prompt"].append([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": s["observation"]}])

train_dataset = Dataset.from_dict(dataset_dict)

In [ ]:
training_args = GRPOConfig(
    learning_rate = 5e-6,
    num_generations = 4, 
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    max_completion_length = 512,
    num_train_epochs = 1,
    output_dir = "outputs",
    report_to = "wandb",
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [
        rule_reward,
        spatial_reward,
        pacing_reward
    ],
    args = training_args,
    train_dataset = train_dataset,
)

trainer.train()